# S&P500/VIX Report Figures

Report-only notebook for CS673/public figures. It does not train models by default. It resolves the first available local output path for the public discrete baseline, the best discrete research model, and the continuous reference, then displays summaries and figures when those local `outputs/` paths exist.

## Setup


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

PUBLIC_BASELINE_OUTPUT_DIR_CANDIDATES = [
    Path(
        "outputs/per_experiment_final_evaluation/sp500_vix/conditional_standard_vq_additive_ar/path_metrics"
    ),
    Path("outputs/sp500_vix_discrete/paper_style_vix_only_temp08_topk40"),
    Path("outputs/sp500_vix_discrete/paper_style"),
]
BEST_DISCRETE_OUTPUT_DIR_CANDIDATES = [
    Path(
        "outputs/per_experiment_final_evaluation/sp500_vix/conditional_hidden128_conv_transformer_k3/path_metrics"
    ),
    Path(
        "outputs/sp500_vix_discrete/paper_style_hidden128_conv_transformer_sampling_temp10_topknone"
    ),
    Path("outputs/sp500_vix_discrete/paper_style_hidden128_conv_transformer_temp08_topk20"),
    Path("outputs/sp500_vix_discrete/best_discrete_research/paper_style"),
]
CONTINUOUS_OUTPUT_DIR_CANDIDATES = [
    Path("outputs/sp500_vix_continuous/beta_cvae/evaluation_final"),
    Path("outputs/per_experiment_final_evaluation/sp500_vix/continuous"),
    Path("outputs/legacy_continuous_evaluation/sp500_vix"),
    Path("outputs/legacy_continuous_evaluation"),
]
PUBLIC_BASELINE_OUTPUT_DIR = PUBLIC_BASELINE_OUTPUT_DIR_CANDIDATES[0]
BEST_DISCRETE_OUTPUT_DIR = BEST_DISCRETE_OUTPUT_DIR_CANDIDATES[0]
CONTINUOUS_OUTPUT_DIR = CONTINUOUS_OUTPUT_DIR_CANDIDATES[0]
RUN_IF_MISSING = False
AUTO_SELECT_MODEL = True
MODEL_REGISTRY_PATH = "trained_models/model_registry.yaml"
MODEL_SELECTION_PROFILE = "balanced_market"
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_HEAVY = False
PROFILE_BATCH_SIZE = 64
PROFILE_WARMUP_RUNS = 1
PROFILE_REPEATS = 3
PROFILE_DEVICE = "cpu"
SUMMARY_FILENAMES = (
    "paper_style_summary.json",
    "summary.json",
    "score_prior_evaluation_summary.json",
)


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path | None) -> Path:
    if path is None:
        raise ValueError("Cannot resolve a null path.")
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path | None) -> str:
    if path is None:
        return "missing"
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_yaml(path: str | Path) -> dict[str, Any]:
    resolved = repo_path(path)
    if not resolved.exists():
        return {}
    loaded = yaml.safe_load(resolved.read_text())
    return loaded if isinstance(loaded, dict) else {}


def load_json(path: str | Path | None) -> dict[str, Any] | None:
    if path is None:
        return None
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


def resolve_summary_path(output_path: Path | None) -> Path | None:
    if output_path is None:
        return None
    resolved = repo_path(output_path)
    if resolved.is_file() and resolved.suffix == ".json":
        return resolved
    if not resolved.exists():
        return None
    for filename in SUMMARY_FILENAMES:
        candidate = resolved / filename
        if candidate.exists():
            return candidate
    nested_sp500_summary = resolved / "sp500_vix" / "summary.json"
    if nested_sp500_summary.exists():
        return nested_sp500_summary
    return None


def resolve_output_entry(entry: dict[str, Any]) -> dict[str, Any]:
    candidates = list(entry.get("path_candidates") or [entry["path"]])
    resolved_path = None
    for candidate in candidates:
        if repo_path(candidate).exists():
            resolved_path = candidate
            break
    resolved = dict(entry)
    resolved["path_candidates"] = candidates
    resolved["resolved_path"] = resolved_path
    resolved["path"] = resolved_path or candidates[0]
    resolved["resolved_summary_path"] = resolve_summary_path(resolved_path)
    return resolved


def resolve_output_dirs(entries: list[dict[str, Any]]) -> list[dict[str, Any]]:
    resolved_entries = [resolve_output_entry(entry) for entry in entries]
    for entry in resolved_entries:
        if entry["resolved_path"] is None:
            candidate_list = ", ".join(display_path(path) for path in entry["path_candidates"])
            print(f"{entry['label']}: skipped; no candidate output path exists ({candidate_list}).")
            continue
        print(f"{entry['label']}: resolved output path -> {display_path(entry['resolved_path'])}")
        if entry["resolved_summary_path"] is not None:
            print(f"{entry['label']}: summary -> {display_path(entry['resolved_summary_path'])}")
        else:
            print(f"{entry['label']}: no recognised summary JSON under the resolved path.")
    return resolved_entries


OUTPUT_DIRS = [
    {
        "key": "public_baseline",
        "label": "Public discrete baseline",
        "path": PUBLIC_BASELINE_OUTPUT_DIR,
        "path_candidates": PUBLIC_BASELINE_OUTPUT_DIR_CANDIDATES,
        "comparison_key": "discrete",
        "model": "standard VQ + additive AR",
    },
    {
        "key": "best_discrete_research",
        "label": "Best discrete research model",
        "path": BEST_DISCRETE_OUTPUT_DIR,
        "path_candidates": BEST_DISCRETE_OUTPUT_DIR_CANDIDATES,
        "comparison_key": "discrete",
        "model": "hidden128 VQ + causal conv-transformer k3",
    },
    {
        "key": "continuous_reference",
        "label": "Continuous BetaCVAE reference",
        "path": CONTINUOUS_OUTPUT_DIR,
        "path_candidates": CONTINUOUS_OUTPUT_DIR_CANDIDATES,
        "comparison_key": "continuous",
        "model": "continuous BetaCVAE",
    },
]
OUTPUT_DIRS = resolve_output_dirs(OUTPUT_DIRS)

PAPER_FIGURES = [
    ("real_vs_generated_paths.png", "Real vs generated S&P500/VIX paths."),
    ("returns_distribution.png", "One-step return distribution."),
    ("terminal_return_distribution.png", "Terminal-return distribution."),
    ("volatility_distribution.png", "Path-volatility distribution."),
    ("maximum_drawdown_distribution.png", "Maximum-drawdown distribution."),
    ("squared_return_autocorrelation.png", "Squared-return autocorrelation."),
    ("vix_bucket_paths.png", "Generated paths by VIX bucket."),
    ("vix_bucket_terminal_returns.png", "VIX-bucket terminal-return comparison."),
    ("vix_bucket_volatility.png", "VIX-bucket volatility comparison."),
]

print(f"Repository root: {REPO_ROOT}")
print("Resolved paths are selected from local candidate lists; no training or evaluation is run.")

## Registered Model Selection


In [ ]:
from time_causal_vae.evaluation.model_profile import ModelProfileSpec, profile_model_specs
from time_causal_vae.experiments.model_registry import load_registry, select_registered_model

REPORT_REGISTRY_ROWS = []
if AUTO_SELECT_MODEL:
    registry = load_registry(repo_path(MODEL_REGISTRY_PATH))
    sp500_discrete_candidates = registry["experiments"]["sp500_vix"]["discrete"]["candidates"]
    public_discrete = select_registered_model(
        registry,
        experiment="sp500_vix",
        family="discrete",
        profile=MODEL_SELECTION_PROFILE,
    )
    continuous_reference = select_registered_model(
        registry,
        experiment="sp500_vix",
        family="continuous",
        profile=MODEL_SELECTION_PROFILE,
    )
    research_candidate_id = "conditional_hidden128_conv_transformer_k3"
    research_candidate = sp500_discrete_candidates.get(research_candidate_id)
    OUTPUT_DIRS = [
        {
            "key": "public_baseline",
            "label": "Public discrete baseline",
            "path": PUBLIC_BASELINE_OUTPUT_DIR,
            "path_candidates": PUBLIC_BASELINE_OUTPUT_DIR_CANDIDATES,
            "comparison_key": "discrete",
            "model": public_discrete.candidate_id,
            "registry_metrics": public_discrete.metrics,
            "missing_metrics": public_discrete.missing_metrics,
            "tokenizer_config": public_discrete.tokenizer_config,
            "prior_config": public_discrete.prior_config,
            "sampling": public_discrete.sampling,
        },
        {
            "key": "continuous_reference",
            "label": "Continuous reference",
            "path": CONTINUOUS_OUTPUT_DIR,
            "path_candidates": CONTINUOUS_OUTPUT_DIR_CANDIDATES,
            "comparison_key": "continuous",
            "model": continuous_reference.candidate_id,
            "registry_metrics": continuous_reference.metrics,
            "missing_metrics": continuous_reference.missing_metrics,
            "config": continuous_reference.config,
        },
    ]
    if research_candidate is not None:
        OUTPUT_DIRS.insert(
            1,
            {
                "key": "best_discrete_research",
                "label": "Best discrete research model",
                "path": BEST_DISCRETE_OUTPUT_DIR,
                "path_candidates": BEST_DISCRETE_OUTPUT_DIR_CANDIDATES,
                "comparison_key": "discrete",
                "model": research_candidate_id,
                "registry_metrics": research_candidate.get("metrics", {}),
                "missing_metrics": research_candidate.get("missing_metrics", []),
                "tokenizer_config": research_candidate.get("tokenizer_config"),
                "prior_config": research_candidate.get("prior_config"),
                "sampling": research_candidate.get("sampling"),
            },
        )
    OUTPUT_DIRS = resolve_output_dirs(OUTPUT_DIRS)
    for entry in OUTPUT_DIRS:
        row = {
            "role": entry["label"],
            "model": entry["model"],
            "output_dir": display_path(entry.get("resolved_path")),
            "summary_path": display_path(entry.get("resolved_summary_path")),
            "path_status": "resolved" if entry.get("resolved_path") is not None else "skipped",
        }
        row.update(entry.get("registry_metrics", {}))
        REPORT_REGISTRY_ROWS.append(row)
    display(pd.DataFrame(REPORT_REGISTRY_ROWS))
else:
    print("AUTO_SELECT_MODEL=False; using notebook-local report defaults.")

## Parameter Count and Inference-Time Profile

The table below profiles the report-comparison architectures from their selected YAML configs. The timing benchmark uses randomly initialised weights on the configured device, so it measures architecture cost rather than checkpoint quality.


In [ ]:
profile_specs = []
for entry in OUTPUT_DIRS:
    if entry["comparison_key"] == "continuous" and entry.get("config") is not None:
        profile_specs.append(
            ModelProfileSpec(
                role=entry["label"],
                family="continuous",
                model=entry["model"],
                config=entry.get("config"),
            )
        )
    elif (
        entry["comparison_key"] == "discrete"
        and entry.get("tokenizer_config") is not None
        and entry.get("prior_config") is not None
    ):
        profile_specs.append(
            ModelProfileSpec(
                role=entry["label"],
                family="discrete",
                model=entry["model"],
                tokenizer_config=entry.get("tokenizer_config"),
                prior_config=entry.get("prior_config"),
                sampling=entry.get("sampling"),
            )
        )

if profile_specs:
    profile_table = pd.DataFrame(
        profile_model_specs(
            profile_specs,
            repo_root=REPO_ROOT,
            batch_size=PROFILE_BATCH_SIZE,
            warmup_runs=PROFILE_WARMUP_RUNS,
            repeats=PROFILE_REPEATS,
            device=PROFILE_DEVICE,
        )
    )
    display(profile_table)
else:
    display(Markdown("No registered model configs were available for profiling."))

## Output Manifest


In [ ]:
inventory_rows = []
for entry in OUTPUT_DIRS:
    output_dir = entry.get("resolved_path")
    summary_path = entry.get("resolved_summary_path")
    inventory_rows.append({
        "role": entry["label"],
        "model": entry["model"],
        "output_dir": display_path(output_dir),
        "summary_path": display_path(summary_path),
        "summary_exists": summary_path is not None and repo_path(summary_path).exists(),
        "candidate_count": len(entry.get("path_candidates", [])),
    })
display(pd.DataFrame(inventory_rows))

for entry, row in zip(OUTPUT_DIRS, inventory_rows, strict=True):
    if row["summary_exists"]:
        continue
    if entry.get("resolved_path") is None:
        candidate_list = ", ".join(display_path(path) for path in entry.get("path_candidates", []))
        print(f"{row['role']}: skipped because no candidate output path exists ({candidate_list}).")
    else:
        print(
            f"{row['role']}: resolved {row['output_dir']} but no recognised summary JSON was found."
        )

if RUN_IF_MISSING:
    print(
        "RUN_IF_MISSING=True is intentionally non-executing in this report notebook; "
        "run the paper-style evaluation commands outside the notebook after replacing "
        "local checkpoint paths."
    )

## Metric Comparison


In [ ]:
PROFILE_KEYS = ["mmd", "swd", "terminal_return_wasserstein", "volatility_wasserstein"]
METRIC_KEYS = [
    "mmd",
    "swd",
    "terminal_return_wasserstein",
    "volatility_wasserstein",
    "maximum_drawdown_wasserstein",
    "squared_return_autocorrelation_within_path_l1",
]


def select_metrics(
    payload: dict[str, Any], preferred_key: str, label: str
) -> tuple[dict[str, Any], str | None]:
    comparisons = payload.get("comparisons", {})
    if isinstance(comparisons, dict) and comparisons:
        if preferred_key in comparisons and isinstance(comparisons[preferred_key], dict):
            return comparisons[preferred_key], preferred_key
        fallback_key = next(iter(comparisons))
        print(f"{label}: comparison `{preferred_key}` missing; using `{fallback_key}` instead.")
        fallback = comparisons[fallback_key]
        return fallback if isinstance(fallback, dict) else {}, fallback_key
    hyper_metric = payload.get("hyper_metric")
    if isinstance(hyper_metric, dict):
        return hyper_metric, "hyper_metric"
    direct_metrics = {key: payload.get(key) for key in METRIC_KEYS if key in payload}
    if direct_metrics:
        return direct_metrics, "summary"
    print(f"{label}: summary has no recognised comparison metrics.")
    return {}, None


summary_rows = []
for entry in OUTPUT_DIRS:
    summary_path = entry.get("resolved_summary_path")
    payload = load_json(summary_path)
    if payload is None:
        continue
    metrics, used_key = select_metrics(payload, entry["comparison_key"], entry["label"])
    if not metrics:
        continue
    profile = None
    if all(key in metrics for key in PROFILE_KEYS):
        profile = sum(float(metrics[key]) for key in PROFILE_KEYS)
    row = {
        "role": entry["label"],
        "model": entry["model"],
        "comparison": used_key,
        "summary_path": display_path(summary_path),
        "profile": profile,
    }
    row.update({key: metrics.get(key) for key in METRIC_KEYS})
    summary_rows.append(row)

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print("No resolved summaries contained recognised report metrics.")

## Display Existing Figures

Figures are displayed only when a resolved local output directory contains them. Summary-only outputs are kept in the metric inventory without failing the notebook.

In [ ]:
for entry in OUTPUT_DIRS:
    output_dir = entry.get("resolved_path")
    if output_dir is None:
        print(f"{entry['label']}: skipped because no candidate output path exists.")
        continue
    resolved_output_dir = repo_path(output_dir)
    if not resolved_output_dir.exists():
        print(f"{entry['label']}: skipped because {display_path(output_dir)} does not exist.")
        continue
    display(Markdown(f"### {entry['label']}: `{display_path(output_dir)}`"))
    figure_rows = []
    for filename, caption in PAPER_FIGURES:
        path = Path(output_dir) / filename
        resolved = repo_path(path)
        if resolved.exists():
            figure_rows.append((filename, caption, resolved))
    if not figure_rows:
        print(f"{entry['label']}: no paper-style figures found under the resolved path.")
        continue
    for filename, caption, resolved in figure_rows:
        display(Markdown(f"#### `{filename}`"))
        display(Markdown(caption))
        display(Image(filename=str(resolved)))

## Output Path Notes

The notebook does not train or evaluate by default. Update the candidate path lists in the setup cell when adding new local report outputs.

In [ ]:
expected_outputs = [
    (entry["label"], entry.get("resolved_path"), entry.get("resolved_summary_path"))
    for entry in OUTPUT_DIRS
]
for label, output_path, summary_path in expected_outputs:
    if output_path is None:
        print(f"{label}: skipped; no candidate output path exists.")
        continue
    print(f"{label}: using {display_path(output_path)}")
    if summary_path is not None:
        print(f"{label}: found summary {display_path(summary_path)}")
    else:
        print(f"{label}: no recognised summary JSON found under resolved path.")

print(
    "The report pass resolves available final-evaluation summaries and local paper-style outputs. "
    "Older default report paths are kept only as low-priority candidates and are not required "
    "when a preferred fallback exists."
)
if not RUN_IF_MISSING:
    print("RUN_IF_MISSING=False; no generation commands are executed from this notebook.")

## Interpretation

Standard VQ with the additive AR prior remains the public discrete default because it gives broad code utilisation, VIX-sensitive code usage, and the simplest one-code-per-time-step interface for the additive scalar-conditioned causal AR prior.

Hidden128 VQ with a causal conv-transformer k3 prior is the best discrete research model under the current S&P500/VIX paper-style profile evidence. Report it as a research variant, not as the new public default, and compare it against the continuous BetaCVAE reference.
